# Обучение агента в Unity ML-Agents с экспортом ONNX

Этот ноутбук обучает агента из Unity-среды (`MyAgent?team=0`) с 6 наблюдениями и 2 непрерывными действиями (без прыжка). Используем PPO из `stable-baselines3` с поддержкой GPU и экспортируем модель в ONNX для Unity Sentis.

## Зависимости
- Установите библиотеки:
  ```bash
  pip install mlagents==0.30.0 stable-baselines3==2.0.0 gymnasium==0.28.1 torch==2.0.1+cu118 torchvision==0.15.2+cu118 -f https://download.pytorch.org/whl/torch_stable.html onnx numpy psutil
  ```
- Убедитесь, что путь к `UnityEnvironment.exe` правильный.
- Среда Unity должна быть собрана с `Behavior Name: MyAgent?team=0`, `Continuous Actions: 2`, `Discrete Actions: 0`.
- Python 3.8, NVIDIA GPU с CUDA 11.8 (или совместимая версия).
- Проверьте доступность GPU:
  ```bash
  python -c "import torch; print(torch.cuda.is_available())"
  ```

In [1]:
!pip3 install mlagents==0.30.0 stable-baselines3==2.0.0 gymnasium==0.28.1 torch==2.0.1+cu118 torchvision==0.15.2+cu118 -f https://download.pytorch.org/whl/torch_stable.html

Looking in links: https://download.pytorch.org/whl/torch_stable.html


In [2]:
# Импорт библиотек
import os
import numpy as np
import torch
import torch.nn as nn
from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.side_channel.engine_configuration_channel import EngineConfigurationChannel
from mlagents_envs.base_env import ActionTuple
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.env_checker import check_env
import gymnasium as gym
from gymnasium import spaces

# Функция закрытия среды
def close_unity_env(env):
    try:
        if env is not None:
            env.close()
            print('Среда Unity успешно закрыта.')
        else:
            print('Среда не инициализирована.')
    except Exception as e:
        print(f'Ошибка при закрытии среды: {e}')
    finally:
        import psutil
        for proc in psutil.process_iter(['name']):
            if proc.info['name'].lower() == 'unityenvironment.exe':
                proc.kill()
                print(f'Процесс UnityEnvironment.exe (PID: {proc.pid}) принудительно завершён.')
            if 'unity' in proc.info['name'].lower():
                proc.kill()
                print(f"Процесс {proc.info['name']} (PID: {proc.pid}) принудительно завершён.")

# Путь к среде Unity
env_path = os.path.join(os.getcwd(), r'N:\MyRL\My_First_NPC\MyfirstMPC\UnityEnvironment.exe')

# Настройка канала для ускорения симуляции
engine_channel = EngineConfigurationChannel()
engine_channel.set_configuration_parameters(time_scale=20.0, quality_level=0) # Уменьшил для стабильности

# Инициализация среды
try:
    env = UnityEnvironment(file_name=env_path, worker_id=1, base_port=6000, side_channels=[engine_channel], timeout_wait=60)
    env.reset()
except Exception as e:
    print(f'Ошибка инициализации среды: {e}')
    close_unity_env(None)
    raise

# Получение имени поведения
behavior_name = list(env.behavior_specs.keys())[0]
print(f'Behavior Name: {behavior_name}')
spec = env.behavior_specs[behavior_name]

# Проверка спецификации
print(f'Observation size: {spec.observation_specs[0].shape[0]}')
print(f'Continuous action size: {spec.action_spec.continuous_size}')
print(f'Discrete action branches: {spec.action_spec.discrete_branches}')

Behavior Name: MyAgent?team=0
Observation size: 6
Continuous action size: 2
Discrete action branches: ()


In [3]:
class CustomActorCriticNet(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=128):
        super(CustomActorCriticNet, self).__init__(observation_space, features_dim)
        self.fc1 = nn.Linear(observation_space.shape[0], 128).to(device)
        self.fc2 = nn.Linear(128, 64).to(device)
        self.fc3 = nn.Linear(64, features_dim).to(device)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

policy_kwargs = dict(
    features_extractor_class=CustomActorCriticNet,
    features_extractor_kwargs=dict(features_dim=128),
    net_arch=[dict(pi=[64, 32], vf=[64, 32])]
)

In [4]:
class UnityGymWrapper(gym.Env):
    def __init__(self, unity_env, behavior_name, spec):
        super(UnityGymWrapper, self).__init__()
        self.env = unity_env
        self.behavior_name = behavior_name
        self.spec = spec
        
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(spec.observation_specs[0].shape[0],), dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(spec.action_spec.continuous_size,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.env.reset()
        decision_steps, _ = self.env.get_steps(self.behavior_name)
        obs = decision_steps.obs[0][0]
        info = {}
        return obs, info

    def step(self, action):
        # Ensure action is shaped (1, action_size)
        action = np.array(action, dtype=np.float32).reshape(1, -1)
        action_tuple = ActionTuple()
        action_tuple.add_continuous(action)  # Continuous actions only

        self.env.set_actions(self.behavior_name, action_tuple)
        self.env.step()

        decision_steps, terminal_steps = self.env.get_steps(self.behavior_name)
        done = len(terminal_steps) > 0
        if done:
            reward = float(terminal_steps.reward[0])
            obs = terminal_steps.obs[0][0]
        else:
            reward = float(decision_steps.reward[0])
            obs = decision_steps.obs[0][0]
        info = {}
        truncated = False

        return obs, reward, done, truncated, info

    def close(self):
        close_unity_env(self.env)

gym_env = UnityGymWrapper(env, behavior_name, spec)
check_env(gym_env)

In [5]:
try:
    # Проверяем доступность GPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Обучение на устройстве: {device}")

    # Инициализируем модель с указанием устройства
    model = PPO(
        'MlpPolicy',
        gym_env,
        policy_kwargs=policy_kwargs,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        verbose=1,
        device=device
    )

    # Обучение
    model.learn(total_timesteps=100000)
    model.save('ppo_myagent_gpu')

    # Выводим информацию о GPU (если используется)
    if device == 'cuda':
        print(f"Используется GPU: {torch.cuda.get_device_name(0)}")
        print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

except Exception as e:
    print(f'Ошибка обучения: {e}')
finally:
    close_unity_env(env)

Обучение на устройстве: cuda
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\Users\User\.conda\envs\ML_Agents\lib\site-packages\stable_baselines3\common\policies.py:460: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 18.9     |
|    ep_rew_mean     | 0.261    |
| time/              |          |
|    fps             | 185      |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 18.2       |
|    ep_rew_mean          | 0.325      |
| time/                   |            |
|    fps                  | 164        |
|    iterations           | 2          |
|    time_elapsed         | 24         |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.00722898 |
|    clip_fraction        | 0.0617     |
|    clip_range           | 0.2        |
|    entropy_loss         | -2.83      |
|    explained_variance   | -0.298     |
|    learning_rate        | 0.0003     |
|   

In [7]:
import torch
from torch.nn import Parameter

# Кастомная обёртка для экспорта в ONNX с поддержкой Sentis
class WrapperNet(torch.nn.Module):
    def __init__(self, policy, continuous_action_size):
        super(WrapperNet, self).__init__()
        self.policy = policy

        # version_number: MLAgents2_0 = 3
        version_number = torch.tensor([3], dtype=torch.float32).to(device)
        self.version_number = Parameter(version_number, requires_grad=False)

        # memory_size: 0, так как нет RNN
        memory_size = torch.tensor([0], dtype=torch.float32).to(device)
        self.memory_size = Parameter(memory_size, requires_grad=False)

        # continuous_action_output_shape: [2] для 2 непрерывных действий
        continuous_shape = torch.tensor([continuous_action_size], dtype=torch.float32).to(device)
        self.continuous_shape = Parameter(continuous_shape, requires_grad=False)

    def forward(self, obs, mask):
        continuous_actions = self.policy(obs, deterministic=True)[0]
        continuous_actions = torch.mul(continuous_actions, mask)  # Фиктивное умножение
        return continuous_actions, self.continuous_shape, self.version_number, self.memory_size

try:
    policy = model.policy.to(device)
    continuous_action_size = spec.action_spec.continuous_size  # 2
    wrapper_net = WrapperNet(policy, continuous_action_size)
    
    dummy_input = torch.randn(1, spec.observation_specs[0].shape[0]).to(device)  # [1, 6]
    dummy_mask = torch.ones(1, continuous_action_size).to(device)  # [1, 2]
    
    torch.onnx.export(
        wrapper_net,
        (dummy_input, dummy_mask),
        'trained_myagent.onnx',
        input_names=['obs_0', 'action_masks'],
        output_names=['continuous_actions', 'continuous_action_output_shape', 'version_number', 'memory_size'],
        dynamic_axes={
            'obs_0': {0: 'batch'},
            'action_masks': {0: 'batch'},
            'continuous_actions': {0: 'batch'},
            'continuous_action_output_shape': {0: 'batch'},
            'version_number': {0: 'batch'},
            'memory_size': {0: 'batch'}
        },
        opset_version=9,
        verbose=False
    )
    print('Модель успешно сохранена: trained_myagent.onnx')
    print('Файл существует:', os.path.exists('trained_myagent.onnx'))
except Exception as e:
    print(f'Ошибка экспорта ONNX: {e}')
    print('Попробуйте opset_version=11 или проверьте версию Unity Sentis.')
finally:
    close_unity_env(env)

============= Diagnostic Run torch.onnx.export version 2.0.1+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

Модель успешно сохранена: trained_myagent.onnx
Файл существует: True
Ошибка при закрытии среды: No Unity environment is loaded.


In [8]:
try:
    obs, _ = gym_env.reset()
    for _ in range(1000):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = gym_env.step(action)
        if done or truncated:
            print('Эпизод завершён')
            obs, _ = gym_env.reset()
except Exception as e:
    print(f'Ошибка тестирования: {e}')
finally:
    close_unity_env(env)

Ошибка тестирования: No Unity environment is loaded.
Ошибка при закрытии среды: No Unity environment is loaded.
